# Week 3 Exercise Solution - Synthetic Data Generation

Using Hugging Face's `datasets` and `models` library, we can create synthetic data.
Formats:
- JSON
- CSV
- JSONL
- Raw Text
- Code (Python Test Cases)

Prompts users can use to generate synthetic data:
- Generate synthetic data points for a dataset of customer reviews for a product. Fields: customer_id, product_id, rating, review.
- Generate employee records for a company. Fields: name, email, phone, department, salary.
- Generate records of sales data for a company. Fields: customer_id, product_id, quantity, amount.
- Generate records of weather data for a city. Fields: date, temperature, precipitation, wind_speed.
- Generate records of stock data for a company. Fields: date, open, high, low, close.
- Generate records of weather data for a city. Fields: date, temperature, precipitation, wind_speed.

In [ ]:
# Instal dependencies
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [ ]:
# Imports
import os
from typing import List, Dict
from dotenv import load_dotenv
from transformers import pipeline
import gradio as gr
import torch


In [ ]:
# Load environment variables
load_dotenv(override=True)

hf_token = os.getenv('HF_TOKEN')
if hf_token:
    print(f"✓ HuggingFace Token exists: {hf_token[:8]}...")
else:
    print("✗ HuggingFace Token not set - some models may not work")
    print("  Get your token from https://huggingface.co/settings/tokens")

In [ ]:
models = [
  "openai/gpt-oss-20b", 
  "microsoft/Phi-4-mini-instruct", 
  "meta-llama/Llama-3.2-3B-Instruct", 
  "Qwen/Qwen3-Coder-Next-FP8"
]

In [ ]:
# Prompts
FORMAT_RULES = {
    "JSON": "Return a valid JSON array. Each element is an object with consistent keys. Start with [ and end with ]. Use double quotes. No markdown fences, no preamble.",
    "CSV": "Return CSV. First line is header with column names. One data row per record. Use commas. Quote fields containing commas.",
    "JSONL": "Return one valid JSON object per line. No blank lines. Each line is self-contained.",
    "Raw Text": "Return prose entries separated by blank lines. Each entry describes one record.",
    "Code": "Return Python code in a singleython fenced block. Use pytest or unittest style test cases."
}



def build_system_prompt(format: str, record_count: int = 5) -> str:
  format_instruction = FORMAT_RULES.get(format, FORMAT_RULES["JSON"])
  format = f"""
  You generate synthetic datasets. Output must be high quality, diverse, and free of PII.
  Constraints:
    - Consistent structure across all records
    - Output ONLY the data—no explanations, no "Here is the JSON/CSV:", no extra text
    - Record count: {record_count}

  Output format: {format}
  Format rules: {format_instruction}
  """
  return format




def build_messages(format: str, record_count: int = 5, user_prompt: str = "") -> List[Dict[str, str]]:
  return [
    {"role": "system", "content": build_system_prompt(format, record_count)},
    {"role": "user", "content": user_prompt}
  ]
  
  

In [ ]:
# Load Pipeline
def load_pipeline(model_name: str):
  pipe = pipeline(
    "text-generation",
    model=model_name,
    # device_map="auto",
    device= 0 if torch.cuda.is_available() else -1
  )
  
  return pipe

In [ ]:
# Cache pipeline per model (avoids reloading on every generate)
from functools import lru_cache

@lru_cache(maxsize=2)
def get_pipeline(model_name: str):
    return load_pipeline(model_name)

def run_generation(model_name: str, user_prompt: str, format: str, record_count: int):
    """Generate synthetic data from the selected model and parameters."""
    user_prompt = user_prompt.strip() or "Generate synthetic records with diverse, realistic data."
    # Map UI "Plain Text" to FORMAT_RULES key
    if format == "Plain Text":
        format = "Raw Text"
    
    try:
        pipe = get_pipeline(model_name)
        messages = build_messages(format, record_count, user_prompt)
        prompt = pipe.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        output = pipe(
            prompt,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            return_full_text=False
        )
        text = output[0]["generated_text"].strip()
        return text
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
# Gradio UI
with gr.Blocks(title="Synthetic Data Generator", theme=gr.themes.Soft()) as demo:
    gr.Markdown("## Synthetic Data Generator")
    gr.Markdown("Describe the data you want. Example: *Generate weather data with fields: country, state, temperature, humidity*")
    
    with gr.Row():
        with gr.Column(scale=1):
            prompt_input = gr.Textbox(
                label="What data to generate",
                placeholder="e.g. Generate weather data with fields: country, state, temperature, humidity",
                lines=3
            )
            format_dropdown = gr.Dropdown(
                choices=["JSON", "CSV", "Plain Text", "Python Code", "JSONL"],
                value="JSON",
                label="Output Format"
            )
            record_slider = gr.Slider(1, 5, value=3, step=1, label="Number of Records (max 5)")
            model_dropdown = gr.Dropdown(
                choices=models,
                value=models[1] if len(models) > 1 else models[0],
                label="Model"
            )
            generate_btn = gr.Button("Generate", variant="primary")
        
        with gr.Column(scale=1):
            output_text = gr.Textbox(label="Generated Output", lines=16)
    
    generate_btn.click(
        run_generation,
        inputs=[
            model_dropdown,
            prompt_input,
            format_dropdown,
            record_slider,
        ],
        outputs=output_text
    )

demo.launch(theme=gr.themes.Soft())